In [1]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print("PyTorch Version:", torch.__version__)

CUDA Available: True
GPU Device: NVIDIA L4
PyTorch Version: 2.6.0+cu124


## Text-only forward pass demo

A minimal nanochat-like transformer is implemented below (PyTorch). It demonstrates a text-only forward pass producing logits for each token.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [3]:
class CharTokenizer:
    def __init__(self, texts=None):
        self.vocab = {'<pad>':0, '<unk>':1}
        self.rev_vocab = {0:'<pad>', 1:'<unk>'}
        if texts:
            for t in texts:
                for ch in t:
                    if ch not in self.vocab:
                        idx = len(self.vocab)
                        self.vocab[ch] = idx
                        self.rev_vocab[idx] = ch
    def encode(self, text):
        return [self.vocab.get(ch, 1) for ch in text]
    def decode(self, ids):
        return ''.join(self.rev_vocab.get(i, '?') for i in ids)
    @property
    def vocab_size(self):
        return len(self.vocab)

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, L, D = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_head ** 0.5)
        mask = torch.triu(torch.full((L, L), float('-inf'), device=x.device), diagonal=1)
        scores = scores + mask
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.out(out)

In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [6]:
class NanoChatLike(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=4, n_heads=4, d_ff=512, max_seq_len=256, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.max_seq_len = max_seq_len
    def forward(self, input_ids):
        B, L = input_ids.size()
        if L > self.max_seq_len:
            raise ValueError('Sequence length exceeds model max_seq_len')
        pos_ids = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos_ids)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

In [7]:
sample_text = 'hello world'
tokenizer = CharTokenizer([sample_text])
print('Vocab size:', tokenizer.vocab_size)
model = NanoChatLike(vocab_size=tokenizer.vocab_size, d_model=128, n_layers=2, n_heads=4, d_ff=512, max_seq_len=128).to(device)
input_ids = torch.tensor([tokenizer.encode(sample_text)], dtype=torch.long, device=device)
with torch.no_grad():
    logits = model(input_ids)
print('Logits shape:', logits.shape)
probs = torch.softmax(logits[0, -1], dim=-1)
topk = torch.topk(probs, k=min(5, probs.size(0)))
print('Top tokens (char -> prob):')
for idx, score in zip(topk.indices, topk.values):
    ch = tokenizer.rev_vocab.get(idx.item(), '?')
    print(ch, float(score))

Vocab size: 10
Logits shape: torch.Size([1, 11, 10])
Top tokens (char -> prob):
l 0.23549814522266388
o 0.18456333875656128
w 0.1288536787033081
<pad> 0.0887109637260437
r 0.08240789920091629
